### A5.1.10. Build Polymorphic Serializer Interface

**Problem:**

Build a serializer interface where each data class knows how to serialize and deserialize itself through a common base protocol.

**Explanation:**

A polymorphic serializer defines a common contract: every serializable class implements `serialize()` (returns a dictionary) and a class-level `deserialize(data)` (reconstructs the object from a dictionary). A dispatcher maps type tags to concrete classes so deserialization works without knowing the type in advance.

How to build it:

1. Define a base protocol: every serializable class must have `serialize()` → `dict` and `deserialize(data)` → instance.
2. Each concrete class implements `serialize` by returning a dict with a `"type"` tag and its fields.
3. Each concrete class implements `deserialize` as a `@classmethod` that reads the dict and constructs an instance.
4. Build a dispatcher: a dict mapping type tags to classes. To deserialize, read the `"type"` tag, look up the class, and call its `deserialize`.

**Example:**

A `Point` and a `Circle` both implement `serialize`/`deserialize`. The dispatcher routes `{"type": "circle", ...}` to `Circle.deserialize()`.

In [ ]:
class Point:
    def __init__(self, x_coord, y_coord):
        self.x_coord = x_coord
        self.y_coord = y_coord

    def serialize(self):
        return {"type": "point", "x": self.x_coord, "y": self.y_coord}

    @classmethod
    def deserialize(cls, data):
        return cls(data["x"], data["y"])


class Circle:
    def __init__(self, center, radius):
        self.center = center
        self.radius = radius

    def serialize(self):
        return {"type": "circle", "center": self.center.serialize(), "radius": self.radius}

    @classmethod
    def deserialize(cls, data):
        center = Point.deserialize(data["center"])
        return cls(center, data["radius"])


TYPE_REGISTRY = {
    "point": Point,
    "circle": Circle,
}


def deserialize(data):
    return TYPE_REGISTRY[data["type"]].deserialize(data)


circle = Circle(Point(1.0, 2.0), 5.0)
serialized = circle.serialize()
print(f"Serialized: {serialized}")

restored = deserialize(serialized)
print(f"Restored: Circle(center=({restored.center.x_coord}, {restored.center.y_coord}), radius={restored.radius})")

**References:**

📘 Gamma, E. et al. (1994). *Design Patterns: Elements of Reusable Object-Oriented Software* — Serialization & Visitor

---

[⬅️ Previous: Build Type Erased Callback Wrapper from Scratch](./09_build_type_erased_callback_wrapper_from_scratch.ipynb) | [Next: Build RAII File Descriptor Wrapper ➡️](./11_build_raii_file_descriptor_wrapper.ipynb)